# Scenario QA dashboard

Lightweight notebook for comparing hazard scenarios during testbed iteration.
Set environment variables (or edit the parameters cell) then run all cells.

- `NIRD_RESULTS_ROOT` — folder with `base_scenario/`, `disruption_analysis/`, etc.
- `NIRD_RESULTS_VARIANT` — e.g. `toy_sioux_falls` or `revision`
- `NIRD_DEPTH_KEY` — flood depth folder (default `30`)
- `NIRD_TESTBED=1` — force KUSD-style cost display
- `NIRD_COST_DISPLAY_UNIT` — `auto`, `kusd`, `musd`, `usd`, or `busd`

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent

viz_dir = root / "scripts" / "visualizations"
if str(viz_dir) not in sys.path:
    sys.path.insert(0, str(viz_dir))

from viz_data_loaders import (
    build_scenario_summary_table,
    format_cost,
    is_testbed_variant,
    list_available_flood_keys,
)


def _first_existing(paths):
    for path in paths:
        if path and Path(path).exists():
            return Path(path)
    return None


results_root = _first_existing([
    os.getenv("NIRD_RESULTS_ROOT"),
    root / "results",
    root / ".pytest-tmp" / "test_sioux_falls_pipeline_scri0" / "results",
])
if results_root is None:
    raise FileNotFoundError("Set NIRD_RESULTS_ROOT to a pipeline results folder.")

variant_candidates = sorted(
    p.name for p in (results_root / "base_scenario").glob("*") if p.is_dir()
) if (results_root / "base_scenario").exists() else []
VARIANT = os.getenv("NIRD_RESULTS_VARIANT") or (variant_candidates[0] if variant_candidates else "revision")
DEPTH_KEY = int(os.getenv("NIRD_DEPTH_KEY", "30"))
FLOOD_KEYS = list_available_flood_keys(results_root, VARIANT, DEPTH_KEY)

print(f"results_root={results_root}")
print(f"variant={VARIANT} testbed={is_testbed_variant(VARIANT)}")
print(f"depth_key={DEPTH_KEY} flood_keys={FLOOD_KEYS}")

## Summary table

In [ ]:
summary = build_scenario_summary_table(
    results_root,
    VARIANT,
    DEPTH_KEY,
    flood_keys=FLOOD_KEYS or None,
)
if summary.empty:
    raise FileNotFoundError(
        f"No disruption outputs under {results_root} for variant={VARIANT}, depth={DEPTH_KEY}"
    )

unit_hint = "KUSD (testbed)" if is_testbed_variant(VARIANT) else "MUSD (production-scale)"
print(f"Cost display: {unit_hint}")
display_cols = [
    "flood_key",
    "flooded_links",
    "closed_links",
    "damaged_links",
    "passenger_disrupted_flow",
    "freight_disrupted_flow",
    "rerouting_cost_passenger_display",
    "rerouting_cost_freight_display",
    "direct_damage_display",
    "combined_total_display",
    "isolation_rows",
    "passenger_flooded_edge_flow",
]
present = [c for c in display_cols if c in summary.columns]
summary[present]

## Scenario comparison charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = summary["flood_key"].astype(str)

axes[0].bar(x, summary["passenger_disrupted_flow"], color="#4c78a8", label="Passenger")
axes[0].bar(x, summary["freight_disrupted_flow"], color="#f58518", alpha=0.7, label="Freight")
axes[0].set_title("Disrupted flow")
axes[0].set_xlabel("Flood event")
axes[0].legend()

axes[1].bar(x, summary["rerouting_cost_passenger_usd"], color="#4c78a8", label="Passenger")
axes[1].bar(x, summary["rerouting_cost_freight_usd"], color="#f58518", alpha=0.7, label="Freight")
axes[1].set_title("Rerouting cost (USD)")
axes[1].set_xlabel("Flood event")
axes[1].legend()

axes[2].bar(x, summary["flooded_links"], color="#54a24b", label="Flooded")
axes[2].bar(x, summary["closed_links"], color="#e45756", alpha=0.7, label="Closed")
axes[2].set_title("Link disruption counts")
axes[2].set_xlabel("Flood event")
axes[2].legend()

fig.suptitle(f"Scenario QA | {VARIANT} | depth={DEPTH_KEY}", fontsize=13)
fig.tight_layout()
plt.show()

## Flooded links map (selected event)

In [ ]:
flood_key = int(os.getenv("NIRD_FLOOD_KEY", str(FLOOD_KEYS[0] if FLOOD_KEYS else 1)))
links_path = (
    results_root
    / "disruption_analysis"
    / VARIANT
    / str(DEPTH_KEY)
    / "links"
    / f"road_links_{flood_key}.gpq"
)
links = gpd.read_parquet(links_path)
links["flood_depth_max"] = pd.to_numeric(links.get("flood_depth_max", 0), errors="coerce").fillna(0)

fig, ax = plt.subplots(figsize=(8, 8))
links.loc[links["flood_depth_max"] <= 0].plot(ax=ax, color="#d3d3d3", linewidth=0.8, label="Dry")
flooded = links.loc[links["flood_depth_max"] > 0]
if not flooded.empty:
    flooded.plot(ax=ax, column="flood_depth_max", cmap="Blues", linewidth=2.0, legend=True)
ax.set_title(f"Flood depth | event={flood_key} | {VARIANT}")
ax.set_axis_off()
plt.show()

## Passenger reroute flow on flooded edges

In [ ]:
reroute_dir = results_root / "rerouting_analysis" / VARIANT / str(DEPTH_KEY) / str(flood_key)
post_path = reroute_dir / "edge_flows_passenger_s1_day1.gpq"
if not post_path.exists():
    print(f"Missing post-reroute flows: {post_path}")
else:
    post = gpd.read_parquet(post_path)
    flow_col = "acc_flow" if "acc_flow" in post.columns else "flow"
    post[flow_col] = pd.to_numeric(post[flow_col], errors="coerce").fillna(0)
    flooded_ids = set(links.loc[links["flood_depth_max"] > 0, "e_id"].astype(str))
    post["layer"] = post["e_id"].astype(str).map(
        lambda e: "flooded" if e in flooded_ids else "other"
    )

    fig, ax = plt.subplots(figsize=(8, 8))
    post.loc[post["layer"] == "other"].plot(ax=ax, color="#d3d3d3", linewidth=0.6)
    subset = post.loc[post["layer"] == "flooded"]
    if not subset.empty:
        subset.plot(ax=ax, column=flow_col, cmap="coolwarm", linewidth=2.0, legend=True)
    ax.set_title(f"Passenger post-reroute flow | flooded edges highlighted")
    ax.set_axis_off()
    plt.show()